# Modélisation – Prédiction du Défaut de Paiement 

**Objectif de cette partie :** entraîner plusieurs modèles, choisir le meilleur via validation croisée, optimiser ses hyperparamètres, choisir un seuil de décision adapté au risque crédit, et évaluer le modèle final sur le jeu de test.

**Plan :**
1. Restauration des données préparées dans le Notebook 2
2. Entraînement de modèles de référence
3. Sélection du meilleur modèle via validation croisée (pas sur le test !)
4. Optimisation des hyperparamètres
5. Choix d'un seuil de décision adapté au risque crédit
6. Évaluation finale sur le test (une seule fois)
7. Importance des variables
8. Conclusion générée automatiquement
9. Lexique complet

## 1. Restauration des données et imports

On récupère les objets `X_train`, `X_test`, `y_train`, `y_test` et `preprocessor` calculés dans le premier notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
                             roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix,
                             classification_report)

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Restauration des variables du Notebook 1
%store -r X_train
%store -r X_test
%store -r y_train
%store -r y_test
%store -r preprocessor
%store -r num_features
%store -r cat_features

print("Données restaurées avec succès !")
print(f"Taille du train : {len(X_train)} | Taille du test  : {len(X_test)}")

**NB :**
- `StratifiedKFold`, `cross_val_score`, `cross_val_predict` : outils de validation croisée, utilisés pour choisir le meilleur modèle et pour calibrer le seuil de décision **sans toucher au test**.
- `precision_recall_curve`, `fbeta_score` : utilisés pour choisir un seuil de décision adapté à notre problème (voir §5).

## 2. Modèles de référence

On entraîne plusieurs algorithmes de classification, du plus simple au plus sophistiqué. Chacun sera enveloppé dans un `Pipeline` avec le `preprocessor` défini ci-dessus : ainsi, l'imputation et l'encodage seront systématiquement (ré)appris sur le train à chaque entraînement, jamais sur le test.

**Petites explications des algorithmes utilisés :**
- **Régression logistique** : trace une frontière (une "ligne") qui sépare au mieux les deux classes. Simple, rapide, facile à interpréter.
- **Random Forest (forêt aléatoire)** : construit de nombreux arbres de décision différents et fait voter la majorité. Robuste, gère bien les interactions complexes.
- **Gradient Boosting / XGBoost / LightGBM / CatBoost** : construisent aussi des arbres, mais **successivement** : chaque nouvel arbre essaie de corriger les erreurs des arbres précédents. Ce sont des variantes optimisées (vitesse, gestion des catégories, etc.) d'une même idée : le "boosting".

**Gestion du déséquilibre de classes :** on a vu dans le notebook 1 qu'il y a beaucoup plus de bons payeurs que de mauvais payeurs. Certains modèles permettent de compenser cela via `class_weight='balanced'` (ou équivalent) : cela dit au modèle de "faire plus attention" aux erreurs sur la classe minoritaire (les défauts) pendant l'entraînement. On l'active partout où c'est disponible.

In [ ]:
# Poids pour compenser le desequilibre de classes, utilise par XGBoost (scale_pos_weight)
poids_classe_positive = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Ratio bons payeurs / mauvais payeurs dans le train : {poids_classe_positive:.2f}")

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    # GradientBoostingClassifier (sklearn) ne supporte pas class_weight nativement.
    # On compensera plutot via le choix du seuil de decision (voir etape 5), qui benefce a tous les modeles.
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss',
                              scale_pos_weight=poids_classe_positive),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1, class_weight='balanced'),
    'CatBoost': CatBoostClassifier(iterations=100, random_seed=42, verbose=False, auto_class_weights='Balanced'),
}

### Comprendre les métriques d'évaluation

Pour un problème comme le nôtre (prédire un défaut, événement plutôt rare), il faut plusieurs métriques complémentaires — l'accuracy seule ne suffit pas (voir notebook 1, §3.1).

Imaginons un tableau à 4 cases, la **matrice de confusion** :

|  | Prédit : pas de défaut | Prédit : défaut |
|---|---|---|
| **Réel : pas de défaut** | Vrai Négatif (VN) ✅ | Faux Positif (FP) ⚠️ |
| **Réel : défaut** | Faux Négatif (FN) 🚨 | Vrai Positif (VP) ✅ |

- **Precision (précision)** : parmi les commerçants que le modèle a signalés "à risque", combien le sont vraiment ? `VP / (VP + FP)`. Une précision élevée = peu de fausses alertes.
- **Recall (rappel)** : parmi les commerçants réellement en défaut, combien le modèle a-t-il su repérer ? `VP / (VP + FN)`. Un rappel élevé = peu de défauts "manqués".
- **F1-score** : moyenne équilibrée entre précision et rappel.
- **ROC-AUC** : mesure la capacité du modèle à bien **classer** les commerçants du plus risqué au moins risqué, quel que soit le seuil choisi. Entre 0.5 (aléatoire) et 1 (parfait).

**Pour un score de risque crédit, quelle métrique privilégier ?** Rater un vrai défaut de paiement (**faux négatif**) coûte généralement plus cher à l'entreprise qu'envoyer une alerte sur un commerçant finalement fiable (**faux positif**, qui coûte surtout du temps de vérification). C'est pourquoi on va porter une attention particulière au **rappel (recall)** dans ce notebook, notamment via le choix du seuil de décision (§5).

### Validation croisée : pourquoi et comment

Au lieu d'évaluer chaque modèle une seule fois sur le test (ce qui ne devrait de toute façon être fait **qu'une fois, à la toute fin**), on utilise la **validation croisée** sur le train pour comparer les modèles entre eux de façon plus fiable.

Le principe (`StratifiedKFold`, 5 "plis") : on découpe le train en 5 portions égales. À tour de rôle, on entraîne sur 4 portions et on évalue sur la 5ème, en changeant la portion "test" à chaque fois. On obtient ainsi 5 scores, dont on prend la moyenne : c'est une estimation bien plus stable qu'un score unique. `Stratified` signifie qu'on garde la même proportion de défauts dans chaque portion.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for name, clf in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])

    # Score de validation croisee (utilise pour comparer/choisir les modeles)
    scores_cv = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

    # Entrainement final sur tout le train, pour pouvoir calculer des metriques detaillees sur le test
    # (uniquement a titre indicatif a ce stade -- ce n'est PAS ce score qui servira a choisir le modele)
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    results[name] = {
        'CV ROC-AUC (moyenne)': scores_cv.mean(),
        'CV ROC-AUC (ecart-type)': scores_cv.std(),
        'Test Accuracy': accuracy_score(y_test, y_pred),
        'Test Precision': precision_score(y_test, y_pred),
        'Test Recall': recall_score(y_test, y_pred),
        'Test F1': f1_score(y_test, y_pred),
        'Test ROC-AUC': roc_auc_score(y_test, y_proba),
        'pipeline': pipeline
    }
    print(f"{name}: CV ROC-AUC = {scores_cv.mean():.4f} (+/- {scores_cv.std():.4f}) | "
          f"Test ROC-AUC (indicatif) = {results[name]['Test ROC-AUC']:.4f}")

⚠️ **Important :** les colonnes "Test ..." ci-dessus sont affichées **à titre indicatif uniquement**, pour la transparence. Elles ne doivent **jamais** servir à choisir le modèle final — sinon, on retombe dans le même problème que "réviser avec le corrigé" (voir notebook 1). C'est le score de validation croisée (**CV ROC-AUC**) qui sert de critère de sélection ci-dessous.

## 3. Sélection des 2 meilleurs modèles (sur la validation croisée, pas sur le test)

On trie les modèles selon leur score de validation croisée moyen, et on garde les 2 meilleurs pour l'étape d'optimisation des hyperparamètres.

In [ ]:
tableau_resultats = pd.DataFrame({
    name: {k: v for k, v in res.items() if k != 'pipeline'}
    for name, res in results.items()
}).T.sort_values('CV ROC-AUC (moyenne)', ascending=False)

display(tableau_resultats)

top2 = tableau_resultats.index[:2].tolist()
print(f"\nTop 2 modeles (selectionnes sur CV ROC-AUC) : {top2}")

## 4. Optimisation des hyperparamètres

Pour chaque modèle du top 2, on teste plusieurs combinaisons de réglages (hyperparamètres) via `RandomizedSearchCV`, qui utilise **déjà** la validation croisée en interne (`cv=cv`) — donc pas de fuite ici non plus.

**Pourquoi "Randomized" et pas "Grid" (grille complète) ?** Tester *toutes* les combinaisons possibles serait très long. `RandomizedSearchCV` en teste un échantillon aléatoire (`n_iter=15` ici), ce qui donne un bon compromis temps/qualité.

*Remarque :* des grilles sont prévues pour tous les modèles de la liste `models`, car selon les données et les réglages, le "top 2" peut varier d'une exécution à l'autre.

In [ ]:
param_grids = {
    'Gradient Boosting': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
        'subsample': [0.8, 1.0]
    },
    'XGBoost': {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.3],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    },
    'LightGBM': {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7, -1],
        'learning_rate': [0.01, 0.05, 0.1],
        'num_leaves': [31, 50, 100],
        'subsample': [0.8, 1.0]
    },
    'CatBoost': {
        'iterations': [100, 200, 300],
        'depth': [4, 6, 8],
        'learning_rate': [0.01, 0.03, 0.1],
        'l2_leaf_reg': [1, 3, 5]
    },
    'Random Forest': {
        'n_estimators': [100, 200, 400],
        'max_depth': [None, 8, 12, 20],
        'min_samples_leaf': [1, 2, 5],
    },
    'Logistic Regression': {
        'C': [0.01, 0.1, 1.0, 10.0],
        'penalty': ['l2'],
    },
}

scores_cv_optimises = {}
optimized = {}
for name in top2:
    clf = models[name]
    pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
    param_grid = {'classifier__' + k: v for k, v in param_grids[name].items()}
    search = RandomizedSearchCV(
        pipeline, param_grid, n_iter=15, cv=cv, scoring='roc_auc', n_jobs=-1, random_state=42
    )
    search.fit(X_train, y_train)
    optimized[name] = search.best_estimator_
    scores_cv_optimises[name] = search.best_score_
    print(f"{name} : meilleur score CV ROC-AUC apres optimisation = {search.best_score_:.4f}")

## 5. Choix du modèle final (toujours sur la validation croisée)

On choisit le modèle final en comparant leurs **meilleurs scores de validation croisée** obtenus après optimisation des hyperparamètres — jamais sur le test.

In [ ]:
best_name = max(scores_cv_optimises, key=scores_cv_optimises.get)
best_pipeline = optimized[best_name]

print(f"Modele final retenu : {best_name}")
print(f"Score CV ROC-AUC (apres optimisation) : {scores_cv_optimises[best_name]:.4f}")

## 6. Choix d'un seuil de décision adapté au risque crédit

### Le problème du seuil par défaut

Par défaut, `.predict()` classe un commerçant comme "à risque" si le modèle estime sa probabilité de défaut **supérieure à 50%**. Mais rien n'oblige à utiliser ce seuil ! Selon le contexte métier, on peut vouloir être plus prudent (baisser le seuil, pour détecter plus de défauts, quitte à avoir plus de fausses alertes) ou au contraire plus sélectif (monter le seuil).

Comme évoqué au §2, dans notre contexte (score de risque crédit), **rater un vrai défaut coûte probablement plus cher qu'une fausse alerte**. On va donc chercher un seuil qui améliore le **rappel (recall)** sans sacrifier excessivement la précision, en utilisant le **F2-score** : une variante du F1-score qui donne 2 fois plus de poids au rappel qu'à la précision.

### Comment choisir ce seuil sans "tricher" avec le test ?

Il serait tentant de tester plusieurs seuils directement sur `X_test` et de garder celui qui donne le meilleur résultat... mais ce serait à nouveau une forme de fuite indirecte : on adapterait notre décision finale en fonction du test, qui doit rester "aveugle" jusqu'au bout.

**Solution : on utilise `cross_val_predict` sur le train.** Cette fonction entraîne le modèle sur 4/5 du train et prédit sur le 1/5 restant, en tournant 5 fois (comme la validation croisée), pour obtenir une probabilité prédite pour **chaque** ligne du train — sans jamais qu'une ligne soit prédite par un modèle qui l'a vue à l'entraînement. C'est donc une estimation honnête, obtenue **sans toucher au test**.

In [ ]:
# Probabilites "hors echantillon" sur le train (chaque ligne predite par un modele qui ne l'a pas vue)
proba_train_oof = cross_val_predict(
    best_pipeline, X_train, y_train, cv=cv, method='predict_proba', n_jobs=-1
)[:, 1]

precisions, recalls, seuils = precision_recall_curve(y_train, proba_train_oof)

# On calcule le F2-score pour chaque seuil candidat (favorise le recall)
# Note : precision_recall_curve renvoie un seuil de moins que precisions/recalls
f2_scores = (5 * precisions[:-1] * recalls[:-1]) / (4 * precisions[:-1] + recalls[:-1] + 1e-10)

meilleur_index = np.argmax(f2_scores)
seuil_optimal = seuils[meilleur_index]

print(f"Seuil par defaut : 0.50")
print(f"Seuil optimal (max F2-score, calcule sur le train uniquement) : {seuil_optimal:.3f}")
print(f"Precision estimee a ce seuil (train, hors echantillon) : {precisions[meilleur_index]:.3f}")
print(f"Recall estimee a ce seuil (train, hors echantillon)    : {recalls[meilleur_index]:.3f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(seuils, precisions[:-1], label='Precision')
plt.plot(seuils, recalls[:-1], label='Recall')
plt.plot(seuils, f2_scores, label='F2-score', linestyle='--')
plt.axvline(seuil_optimal, color='red', linestyle=':', label=f'Seuil retenu ({seuil_optimal:.2f})')
plt.xlabel('Seuil de decision')
plt.ylabel('Score')
plt.title('Compromis precision / rappel selon le seuil (estime sur le train, hors echantillon)')
plt.legend()
plt.show()

## 7. Évaluation finale sur le test (une seule fois)

C'est le **seul** moment de ce notebook où l'on regarde les vraies performances sur `X_test` / `y_test` de façon définitive. On réentraîne le modèle final sur l'intégralité du train, puis on l'applique au test avec les deux seuils (0.5 et le seuil optimisé), pour comparer.

In [ ]:
best_pipeline.fit(X_train, y_train)
y_proba_test = best_pipeline.predict_proba(X_test)[:, 1]

y_pred_default = (y_proba_test >= 0.5).astype(int)
y_pred_optimal = (y_proba_test >= seuil_optimal).astype(int)

comparaison = pd.DataFrame({
    'Seuil = 0.50 (defaut)': {
        'Accuracy': accuracy_score(y_test, y_pred_default),
        'Precision': precision_score(y_test, y_pred_default),
        'Recall': recall_score(y_test, y_pred_default),
        'F1-score': f1_score(y_test, y_pred_default),
        'F2-score': fbeta_score(y_test, y_pred_default, beta=2),
    },
    f'Seuil = {seuil_optimal:.2f} (optimise)': {
        'Accuracy': accuracy_score(y_test, y_pred_optimal),
        'Precision': precision_score(y_test, y_pred_optimal),
        'Recall': recall_score(y_test, y_pred_optimal),
        'F1-score': f1_score(y_test, y_pred_optimal),
        'F2-score': fbeta_score(y_test, y_pred_optimal, beta=2),
    }
}).T

print(f"=== Modele final : {best_name} ===")
print(f"ROC-AUC sur le test (independant du seuil) : {roc_auc_score(y_test, y_proba_test):.4f}\n")
display(comparaison.round(4))

**Comment lire ce tableau ?** Le ROC-AUC ne dépend pas du seuil choisi (il mesure la qualité du classement des probabilités). En revanche, l'accuracy/précision/rappel/F1/F2 changent selon le seuil retenu. On s'attend à ce que le seuil optimisé donne un **meilleur rappel** (moins de défauts manqués), au prix d'une précision un peu plus faible (plus de fausses alertes) — un compromis assumé et justifié par le contexte métier (voir §6).

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba_test)
plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc_score(y_test, y_proba_test):.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Modele aleatoire')
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs (recall)')
plt.title(f'Courbe ROC -- {best_name}')
plt.legend()
plt.show()

sns.heatmap(confusion_matrix(y_test, y_pred_optimal), annot=True, fmt='d', cmap='Blues')
plt.title(f'Matrice de confusion (seuil = {seuil_optimal:.2f})')
plt.xlabel('Prediction')
plt.ylabel('Reel')
plt.show()

print(classification_report(y_test, y_pred_optimal, target_names=['Bon payeur', 'Defaut']))

## 8. Importance des variables

On regarde quelles variables ont le plus influencé les décisions du modèle final (disponible uniquement pour les modèles à base d'arbres, comme Gradient Boosting, XGBoost, LightGBM, CatBoost, Random Forest).

⚠️ Rappel important : l'importance d'une variable indique son **utilité pour la prédiction**, pas nécessairement une relation de cause à effet.

In [ ]:
clf_final = best_pipeline.named_steps['classifier']

if hasattr(clf_final, 'feature_importances_'):
    preprocessor_final = best_pipeline.named_steps['preprocessor']
    cat_names = preprocessor_final.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_features)
    feature_names = num_features + list(cat_names)

    importances = pd.DataFrame({
        'feature': feature_names,
        'importance': clf_final.feature_importances_
    }).sort_values('importance', ascending=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=importances.head(10), x='importance', y='feature')
    plt.title(f'Top 10 variables les plus importantes -- {best_name}')
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_name} ne fournit pas nativement d'importance de variables (ex: regression logistique).")

## 9. Conclusion

La conclusion ci-dessous est générée **automatiquement à partir des résultats réellement calculés** dans ce notebook (et non écrite en dur), pour éviter tout décalage entre le texte et les chiffres.

In [ ]:
print("="*70)
print("RESUME FINAL -- SenCredit, modele de risque de defaut de paiement")
print("="*70)
print(f"\nModele retenu : {best_name}")
print(f"Selectionne sur la base du score de validation croisee (pas du test).")
print(f"\nPerformance de classement (independante du seuil) :")
print(f"  - ROC-AUC (test) : {roc_auc_score(y_test, y_proba_test):.4f}")
print(f"\nPerformance au seuil optimise ({seuil_optimal:.2f}), sur le test :")
for metric, val in comparaison.loc[f'Seuil = {seuil_optimal:.2f} (optimise)'].items():
    print(f"  - {metric} : {val:.4f}")
print(f"\nA titre de comparaison, au seuil par defaut (0.50) :")
for metric, val in comparaison.loc['Seuil = 0.50 (defaut)'].items():
    print(f"  - {metric} : {val:.4f}")
print("\n" + "="*70)

### Interprétation

- Un **ROC-AUC** nettement supérieur à 0.5 indique que le modèle sait globalement bien distinguer les bons payeurs des mauvais, indépendamment du seuil choisi.
- Le passage du seuil par défaut (0.5) au seuil optimisé illustre concrètement le compromis précision/rappel : regardez les chiffres affichés ci-dessus pour voir de combien le rappel a progressé, et à quel coût en précision.
- Les variables les plus importantes (§8) donnent des pistes d'action concrètes pour l'équipe métier (relance ciblée, conditions de crédit adaptées, etc.), à confirmer par une analyse métier complémentaire.

### Recommandations

1. **Utiliser ce modèle comme aide à la décision**, pas comme automatisation totale : un score élevé doit déclencher une vérification humaine, pas un refus automatique.
2. **Valider l'hypothèse sur `Volume_Mensuel_FCFA == 0`** (voir notebook 1, §2.3) auprès du métier avant tout déploiement — si cette hypothèse est fausse, elle peut biaiser le modèle.
3. **Réévaluer régulièrement** le modèle avec des données plus récentes (dérive des comportements dans le temps) et refaire la calibration du seuil si le contexte métier ou le coût des faux négatifs/positifs évolue.
4. **Explorer des données complémentaires** (historique bancaire, données télécom) si disponibles, pour enrichir les prédictions au-delà des variables actuelles.
5. **Ne jamais réutiliser l'ensemble de test** de ce notebook pour de futurs ajustements du modèle : une fois "consommé" pour l'évaluation finale, il doit être considéré comme "vu" et ne plus servir de juge impartial. Pour de futures itérations, prévoir un nouvel échantillon de test.

## 📖 Lexique complet (pour débutants)

| Terme | Définition simple |
|---|---|
| **Apprentissage supervisé** | Apprendre à partir d'exemples déjà étiquetés (ici, des commerçants dont on connaît déjà le statut de défaut). |
| **Train / Test split** | Séparation des données en un ensemble d'entraînement et un ensemble d'évaluation, jamais mélangés. |
| **Fuite de données (leakage)** | Utiliser, même involontairement, une information qu'on n'aurait pas dû avoir, ce qui fausse l'évaluation du modèle. |
| **Pipeline** | Enchaînement d'étapes de transformation + modèle, traité comme un seul bloc reproductible. |
| **Imputation** | Remplacement d'une valeur manquante par une valeur estimée. |
| **One-hot encoding** | Transformation d'une catégorie en plusieurs colonnes 0/1 (une par valeur possible). |
| **Target encoding (encodage par cible)** | Remplacement d'une catégorie par une statistique liée à la cible (ex : taux de défaut moyen de cette catégorie). |
| **Lissage (smoothing)** | Technique pour éviter que des statistiques calculées sur peu d'observations soient trop instables, en les rapprochant d'une moyenne globale. |
| **Validation croisée (cross-validation)** | Évaluer un modèle plusieurs fois sur des portions différentes des données, pour une estimation plus fiable de sa performance. |
| **Hyperparamètre** | Réglage du modèle choisi avant l'entraînement (ex : nombre d'arbres), à ne pas confondre avec ce que le modèle apprend seul. |
| **Déséquilibre de classes** | Une catégorie de la cible bien plus fréquente que l'autre. |
| **Matrice de confusion** | Tableau croisant prédictions et réalité (vrais/faux positifs/négatifs). |
| **Precision (précision)** | Parmi les alertes positives, la proportion qui est correcte. |
| **Recall (rappel)** | Parmi les cas positifs réels, la proportion correctement détectée. |
| **F1-score / F2-score** | Moyennes combinant précision et rappel ; le F2-score donne plus de poids au rappel. |
| **ROC-AUC** | Mesure de la capacité d'un modèle à bien classer les observations, indépendamment du seuil choisi. |
| **Seuil de décision** | Valeur de probabilité au-delà de laquelle on classe une observation comme "positive" (ici, "à risque"). |
| **Overfitting (surapprentissage)** | Quand un modèle "colle" trop aux données d'entraînement (y compris à leur bruit) et généralise mal à de nouvelles données. |

Fin du parcours SénCrédit — vous avez maintenant une base solide pour comprendre un projet de data science de bout en bout, depuis le nettoyage des données jusqu'à un modèle évalué de façon rigoureuse. 🎉